# Earth4D LFMC Ablation Study

**Comprehensive ablation study for LFMC prediction**

This notebook runs 8 experiments to evaluate different feature combinations:

**Without Species Embeddings:**
1. Earth4D alone
2. AEF alone
3. Earth4D + AEF
4. Earth4D + AEF + Daymet

**With Species Embeddings (768D):**
5. Earth4D + Species
6. AEF + Species
7. Earth4D + AEF + Species
8. Earth4D + AEF + Daymet + Species (Full Model)

**Configuration (Updated to match original successful run):**
- **2500 epochs** per experiment (was 100)
- 768-dimension species embeddings
- **Batch size: 1024** (was 30000)
- **Learning rate: 0.001** (was 0.03)
- Random seed: 0
- Automatic dataset download from cloud storage
- Comprehensive visualization with labeled plots
- PNG and SVG outputs (DPI 300)

**Note:** The original ablation had batch_size=30000 and lr=0.03, which caused ~3pp worse performance compared to the original successful run that used batch_size=1024 and lr=0.001. This updated version uses the correct hyperparameters.

## 1. Environment Setup

In [ ]:
# Check GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: CUDA not available - this model requires GPU!")

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
!pip install -q numpy pandas scikit-learn scipy tqdm matplotlib seaborn
!pip install -q ninja pybind11
!pip install -q pyarrow fastparquet  # For Parquet support

print("Dependencies installed!")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/Earth4D_LFMC_Ablation_Study'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

print(f"Working directory: {os.getcwd()}")

## 2. Download Repository (earth-observation branch)

In [ ]:
# Download earth-observation branch from GitHub
import os
import shutil

# Clean previous installation
if os.path.exists('deepearth-earth-observation'):
    shutil.rmtree('deepearth-earth-observation')
if os.path.exists('deepearth-earth-observation.zip'):
    os.remove('deepearth-earth-observation.zip')

# Download earth-observation branch
!wget -O deepearth-earth-observation.zip https://github.com/legel/deepearth/archive/refs/heads/earth-observation.zip
!unzip -q deepearth-earth-observation.zip

print("Repository downloaded!")
print("\nContents:")
!ls -la deepearth-earth-observation/encoders/xyzt/ | head -20

In [ ]:
# Install Earth4D
EARTH4D_DIR = os.path.join(WORK_DIR, 'deepearth-earth-observation', 'encoders', 'xyzt')
os.chdir(EARTH4D_DIR)

print(f"Installing Earth4D from: {os.getcwd()}")
!pip install -e .

print("\nEarth4D installed!")

In [ ]:
# Test Earth4D import
from earth4d import Earth4D
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
earth4d = Earth4D(
    spatial_levels=4,
    temporal_levels=3,
    features_per_level=2,
    verbose=False
).to(device)

test_coords = torch.tensor([[0.0, 0.0, 0.0, 0.5]], device=device)
with torch.no_grad():
    features = earth4d(test_coords)

print(f"Earth4D import successful!")
print(f"Test output shape: {features.shape}")

## 3. Configuration

In [ ]:
# PathsEARTH4D_DIR = os.path.join(WORK_DIR, 'deepearth-earth-observation', 'encoders', 'xyzt')DATA_DIR = os.path.join(WORK_DIR, 'data')OUTPUT_DIR = os.path.join(WORK_DIR, 'ablation_results')os.makedirs(DATA_DIR, exist_ok=True)os.makedirs(OUTPUT_DIR, exist_ok=True)# Global training config - UPDATED TO MATCH MAIN BRANCHEPOCHS = 250  # CHANGED: 2500 -> 250 for initial validationSPECIES_DIM = 768BATCH_SIZE = 30000  # CHANGED: 1024 -> 30000 to match main branch (learned probing)LEARNING_RATE = 0.0125  # CHANGED: 0.001 -> 0.0125 to match main branch (learned probing)SEED = 0print("Configuration (UPDATED to match main branch with learned probing):")print(f"  Epochs: {EPOCHS}")print(f"  Species dim: {SPECIES_DIM}")print(f"  Batch size: {BATCH_SIZE} (from main branch: 30000)")print(f"  Learning rate: {LEARNING_RATE} (from main branch: 0.0125)")print(f"  Random seed: {SEED}")print(f"Directories:")print(f"  Earth4D: {EARTH4D_DIR}")print(f"  Data: {DATA_DIR}")print(f"  Output: {OUTPUT_DIR}")

In [ ]:
# Define 8 ablation experimentsexperiments = [    {        'name': 'exp1_earth4d_only',        'label': 'Earth4D Only',        'use_earth4d': True,        'use_species': False,        'use_aef': False,        'use_daymet': False    },    {        'name': 'exp2_aef_only',        'label': 'AEF Only',        'use_earth4d': False,        'use_species': False,        'use_aef': True,        'use_daymet': False    },    {        'name': 'exp3_earth4d_aef',        'label': 'Earth4D + AEF',        'use_earth4d': True,        'use_species': False,        'use_aef': True,        'use_daymet': False    },    {        'name': 'exp4_earth4d_aef_daymet',        'label': 'Earth4D + AEF + Daymet',        'use_earth4d': True,        'use_species': False,        'use_aef': True,        'use_daymet': True    },    {        'name': 'exp5_earth4d_species',        'label': 'Earth4D + Species',        'use_earth4d': True,        'use_species': True,        'use_aef': False,        'use_daymet': False    },    {        'name': 'exp6_aef_species',        'label': 'AEF + Species',        'use_earth4d': False,        'use_species': True,        'use_aef': True,        'use_daymet': False    },    {        'name': 'exp7_earth4d_aef_species',        'label': 'Earth4D + AEF + Species',        'use_earth4d': True,        'use_species': True,        'use_aef': True,        'use_daymet': False    },    {        'name': 'exp8_full_model',        'label': 'Full Model (Earth4D + AEF + Daymet + Species)',        'use_earth4d': True,        'use_species': True,        'use_aef': True,        'use_daymet': True    }]print("Ablation Experiments:")for i, exp in enumerate(experiments, 1):    features = []    if exp['use_earth4d']: features.append('Earth4D')    if exp['use_species']:         try:            features.append(f'Species({SPECIES_DIM}D)')        except NameError:            features.append('Species(768D)')  # Default value if SPECIES_DIM not defined yet    if exp['use_aef']: features.append('AEF(64D)')    if exp['use_daymet']: features.append('Daymet(21D)')    print(f"  {i}. {exp['label']:45s} = {' + '.join(features)}")

## 4. Run All 8 Ablation Experiments

In [ ]:
import subprocess
import time
from datetime import datetime

# Change to Earth4D directory
os.chdir(EARTH4D_DIR)

results = []

print("="*80)
print("RUNNING 8 ABLATION EXPERIMENTS")
print("="*80)

for i, exp in enumerate(experiments, 1):
    print(f"\n{'='*80}")
    print(f"EXPERIMENT {i}/8: {exp['label']}")
    print(f"{'='*80}")
    
    # Create experiment output directory
    exp_output_dir = os.path.join(OUTPUT_DIR, exp['name'])
    os.makedirs(exp_output_dir, exist_ok=True)
    
    # Build command
    cmd = [
        'python', 'earth4d-aef-daymet_to_lfmc.py',
        '--data-dir', DATA_DIR,
        '--epochs', str(EPOCHS),
        '--species-dim', str(SPECIES_DIM),
        '--batch-size', str(BATCH_SIZE),
        '--lr', str(LEARNING_RATE),
        '--output-dir', exp_output_dir,
        '--seed', str(SEED),
        '--auto-download'  # Auto-download datasets
    ]
    
    # Add feature flags
    if not exp['use_earth4d']:
        cmd.append('--no-earth4d')
    if not exp['use_species']:
        cmd.append('--no-species')
    if exp['use_aef']:
        cmd.append('--use-aef')
    if exp['use_daymet']:
        cmd.append('--use-daymet')
    
    print(f"\nCommand: {' '.join(cmd)}")
    print(f"\nStarting training at {datetime.now().strftime('%H:%M:%S')}...")
    
    start_time = time.time()
    
    # Run experiment
    result = subprocess.run(cmd, capture_output=False)
    
    elapsed = time.time() - start_time
    
    if result.returncode == 0:
        print(f"\n[SUCCESS] Experiment {i} completed in {elapsed/60:.1f} minutes")
        results.append({
            'experiment': exp['name'],
            'label': exp['label'],
            'success': True,
            'time_minutes': elapsed/60
        })
    else:
        print(f"\n[ERROR] Experiment {i} failed!")
        results.append({
            'experiment': exp['name'],
            'label': exp['label'],
            'success': False,
            'time_minutes': elapsed/60
        })

print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETED")
print("="*80)

# Print summary
print("\nSummary:")
for r in results:
    status = "SUCCESS" if r['success'] else "FAILED"
    print(f"  {r['label']:45s} - {status} ({r['time_minutes']:.1f} min)")

n_success = sum(1 for r in results if r['success'])
print(f"\nTotal: {n_success}/{len(results)} experiments successful")

## 5. Collect and Analyze Results

In [ ]:
import pandas as pd
import glob

# Collect all metrics CSVs
all_metrics = {}

for exp in experiments:
    exp_output_dir = os.path.join(OUTPUT_DIR, exp['name'])
    
    # Find metrics CSV
    csv_files = glob.glob(os.path.join(exp_output_dir, 'training_metrics_*.csv'))
    
    if csv_files:
        # Load most recent CSV
        csv_path = sorted(csv_files)[-1]
        df = pd.read_csv(csv_path)
        all_metrics[exp['name']] = {
            'label': exp['label'],
            'data': df
        }
        print(f"Loaded: {exp['label']} ({len(df)} epochs)")
    else:
        print(f"WARNING: No metrics found for {exp['label']}")

print(f"\nLoaded {len(all_metrics)}/8 experiment results")

## 6. Create Comparison Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set high DPI for publication-quality figures
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# Define colors for experiments (without species vs with species)
colors_no_species = ['#E74C3C', '#E67E22', '#F39C12', '#F1C40F']  # Reds/Oranges
colors_with_species = ['#3498DB', '#2ECC71', '#1ABC9C', '#9B59B6']  # Blues/Greens

color_map = {
    'exp1_earth4d_only': colors_no_species[0],
    'exp2_aef_only': colors_no_species[1],
    'exp3_earth4d_aef': colors_no_species[2],
    'exp4_earth4d_aef_daymet': colors_no_species[3],
    'exp5_earth4d_species': colors_with_species[0],
    'exp6_aef_species': colors_with_species[1],
    'exp7_earth4d_aef_species': colors_with_species[2],
    'exp8_full_model': colors_with_species[3]
}

print("Color scheme defined:")
print("  Red/Orange: Experiments WITHOUT species")
print("  Blue/Green: Experiments WITH species")

In [ ]:
# Figure 1: Training Loss Curves (All Experiments)
fig, ax = plt.subplots(figsize=(14, 8))

for exp_name, exp_data in all_metrics.items():
    df = exp_data['data']
    label = exp_data['label']
    color = color_map[exp_name]
    
    # Plot training MAE
    ax.plot(df['epoch'], df['train_mae'], 
            label=label, 
            color=color, 
            linewidth=2.5,
            alpha=0.9)

ax.set_xlabel('Epoch', fontsize=14, fontweight='bold')
ax.set_ylabel('Training MAE (percentage points)', fontsize=14, fontweight='bold')
ax.set_title('Training Loss Curves - 8 Ablation Experiments', 
             fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='upper right', fontsize=11, framealpha=0.95)
ax.grid(True, alpha=0.3, linestyle='--')
ax.set_xlim(0, EPOCHS)

plt.tight_layout()

# Save
png_path = os.path.join(OUTPUT_DIR, 'fig1_training_loss_curves.png')
svg_path = os.path.join(OUTPUT_DIR, 'fig1_training_loss_curves.svg')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, dpi=300, bbox_inches='tight')
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

plt.show()

In [ ]:
# Figure 2: Test Performance (Temporal, Spatial, Random)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

test_sets = [
    ('temporal_mae', 'Temporal Test MAE', axes[0]),
    ('spatial_mae', 'Spatial Test MAE', axes[1]),
    ('random_mae', 'Random Test MAE', axes[2])
]

for metric, title, ax in test_sets:
    for exp_name, exp_data in all_metrics.items():
        df = exp_data['data']
        label = exp_data['label']
        color = color_map[exp_name]
        
        ax.plot(df['epoch'], df[metric], 
                label=label, 
                color=color, 
                linewidth=2.5,
                alpha=0.9)
    
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('MAE (percentage points)', fontsize=12, fontweight='bold')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, framealpha=0.95)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(0, EPOCHS)

fig.suptitle('Test Set Performance - 8 Ablation Experiments', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Save
png_path = os.path.join(OUTPUT_DIR, 'fig2_test_performance.png')
svg_path = os.path.join(OUTPUT_DIR, 'fig2_test_performance.svg')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, dpi=300, bbox_inches='tight')
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

plt.show()

In [ ]:
# Figure 3: Final Performance Comparison (Bar Chart) - FIXED
final_performance = {}

for exp_name, exp_data in all_metrics.items():
    df = exp_data['data']
    final_epoch = df.iloc[-1]
    
    final_performance[exp_name] = {
        'label': exp_data['label'],
        'train_mae': final_epoch['train_mae'],
        'temporal_mae': final_epoch['temporal_mae'],
        'spatial_mae': final_epoch['spatial_mae'],
        'random_mae': final_epoch['random_mae']
    }

# Create bar chart
fig, ax = plt.subplots(figsize=(14, 8))

# FIX: Changed from incorrect iteration to correct iteration over experiments
labels = [final_performance[exp['name']]['label'] for exp in experiments if exp['name'] in final_performance]
train_vals = [final_performance[exp['name']]['train_mae'] for exp in experiments if exp['name'] in final_performance]
temporal_vals = [final_performance[exp['name']]['temporal_mae'] for exp in experiments if exp['name'] in final_performance]
spatial_vals = [final_performance[exp['name']]['spatial_mae'] for exp in experiments if exp['name'] in final_performance]
random_vals = [final_performance[exp['name']]['random_mae'] for exp in experiments if exp['name'] in final_performance]

x = np.arange(len(labels))
width = 0.2

bars1 = ax.bar(x - 1.5*width, train_vals, width, label='Train', color='#34495E', alpha=0.8)
bars2 = ax.bar(x - 0.5*width, temporal_vals, width, label='Temporal Test', color='#E74C3C', alpha=0.8)
bars3 = ax.bar(x + 0.5*width, spatial_vals, width, label='Spatial Test', color='#3498DB', alpha=0.8)
bars4 = ax.bar(x + 1.5*width, random_vals, width, label='Random Test', color='#2ECC71', alpha=0.8)

ax.set_xlabel('Experiment', fontsize=14, fontweight='bold')
ax.set_ylabel('Final MAE (percentage points)', fontsize=14, fontweight='bold')
ax.set_title(f'Final Performance After {EPOCHS} Epochs - 8 Ablation Experiments', 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels([f"{i+1}. {label.split(' (')[0]}" 
                     for i, label in enumerate(labels)], 
                    rotation=45, ha='right', fontsize=10)
ax.legend(loc='upper right', fontsize=12)
ax.grid(True, alpha=0.3, axis='y', linestyle='--')

plt.tight_layout()

# Save
png_path = os.path.join(OUTPUT_DIR, 'fig3_final_performance_comparison.png')
svg_path = os.path.join(OUTPUT_DIR, 'fig3_final_performance_comparison.svg')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, dpi=300, bbox_inches='tight')
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

plt.show()

In [ ]:
# Figure 4: Species vs No-Species Comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

comparisons = [
    ('exp1_earth4d_only', 'exp5_earth4d_species', 'Earth4D', axes[0, 0]),
    ('exp2_aef_only', 'exp6_aef_species', 'AEF', axes[0, 1]),
    ('exp3_earth4d_aef', 'exp7_earth4d_aef_species', 'Earth4D + AEF', axes[1, 0]),
    ('exp4_earth4d_aef_daymet', 'exp8_full_model', 'Earth4D + AEF + Daymet', axes[1, 1])
]

for exp_no_species, exp_with_species, base_label, ax in comparisons:
    if exp_no_species in all_metrics and exp_with_species in all_metrics:
        # Plot without species
        df_no = all_metrics[exp_no_species]['data']
        ax.plot(df_no['epoch'], df_no['temporal_mae'], 
                label=f'{base_label} (No Species)', 
                color=color_map[exp_no_species], 
                linewidth=2.5,
                linestyle='--',
                alpha=0.9)
        
        # Plot with species
        df_with = all_metrics[exp_with_species]['data']
        ax.plot(df_with['epoch'], df_with['temporal_mae'], 
                label=f'{base_label} + Species', 
                color=color_map[exp_with_species], 
                linewidth=2.5,
                alpha=0.9)
        
        ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
        ax.set_ylabel('Temporal Test MAE (pp)', fontsize=12, fontweight='bold')
        ax.set_title(f'{base_label}: Species Impact', fontsize=13, fontweight='bold')
        ax.legend(loc='upper right', fontsize=11)
        ax.grid(True, alpha=0.3, linestyle='--')
        ax.set_xlim(0, EPOCHS)

fig.suptitle('Impact of Species Embeddings (768D) on Different Feature Sets', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()

# Save
png_path = os.path.join(OUTPUT_DIR, 'fig4_species_impact_comparison.png')
svg_path = os.path.join(OUTPUT_DIR, 'fig4_species_impact_comparison.svg')
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, dpi=300, bbox_inches='tight')
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")

plt.show()

In [ ]:
# Helper function for creating short experiment labels
def create_short_label(label):
    """Create short label with feature indicators"""
    features = []
    if 'Earth4D' in label:
        features.append('E4D')
    if 'AEF' in label:
        features.append('AEF')
    if 'Daymet' in label:
        features.append('Day')
    if 'Species' in label:
        features.append('Sp')
    if not features:
        return 'None'
    return '+'.join(features)

print("Helper function defined: create_short_label()")

In [ ]:
# Figure 5: Training vs Validation Loss - All Experiments (Same Y-Scale)import matplotlib.pyplot as pltimport numpy as npfig, axes = plt.subplots(2, 4, figsize=(24, 12))axes = axes.flatten()# Find global y-limits across all experimentsall_train_vals = []all_val_vals = []for exp_name, exp_data in all_metrics.items():    df = exp_data['data']    all_train_vals.extend(df['train_mae'].values)    all_val_vals.extend(df['temporal_mae'].values)y_min = min(min(all_train_vals), min(all_val_vals))y_max = max(max(all_train_vals), max(all_val_vals))# Add 5% padding to y-limitsy_range = y_max - y_miny_min = y_min - 0.05 * y_rangey_max = y_max + 0.05 * y_rangefor idx, exp in enumerate(experiments):    exp_name = exp['name']    if exp_name not in all_metrics:        continue        ax = axes[idx]    df = all_metrics[exp_name]['data']        # Plot train and validation    ax.plot(df['epoch'], df['train_mae'],             label='Training', color='#2C3E50', linewidth=2.5, alpha=0.9)    ax.plot(df['epoch'], df['temporal_mae'],             label='Validation (Temporal)', color='#E74C3C', linewidth=2.5, alpha=0.9)        # Create short label    short_label = create_short_label(all_metrics[exp_name]['label'])        # Set same y-limits for all panels    ax.set_ylim(y_min, y_max)    ax.set_xlim(0, EPOCHS)        # Formatting    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')    ax.set_ylabel('MAE (pp)', fontsize=11, fontweight='bold')    ax.set_title(f"{idx+1}. {short_label}", fontsize=13, fontweight='bold', pad=10)    ax.legend(loc='upper right', fontsize=10, framealpha=0.95)    ax.grid(True, alpha=0.3, linestyle='--')        # Add final values as text    final_train = df['train_mae'].iloc[-1]    final_val = df['temporal_mae'].iloc[-1]    gap = final_val - final_train        textstr = f'Epoch {EPOCHS}:'    textstr += f'Train: {final_train:.2f}'    textstr += f'Val: {final_val:.2f}'    textstr += f'Gap: {gap:+.2f}'        props = dict(boxstyle='round', facecolor='wheat', alpha=0.85)    ax.text(0.97, 0.97, textstr,            transform=ax.transAxes, fontsize=9,            verticalalignment='top', horizontalalignment='right',            bbox=props, family='monospace')fig.suptitle('Training vs Validation Loss - All Experiments (Same Y-Scale)(Same y-axis limits across all panels for easy comparison)',             fontsize=17, fontweight='bold', y=0.995)plt.tight_layout()# Savepng_path = os.path.join(OUTPUT_DIR, 'fig5_train_val_comparison_same_scale.png')svg_path = os.path.join(OUTPUT_DIR, 'fig5_train_val_comparison_same_scale.svg')plt.savefig(png_path, dpi=300, bbox_inches='tight')plt.savefig(svg_path, dpi=300, bbox_inches='tight')print(f"Saved: {png_path}")print(f"Saved: {svg_path}")plt.show()# Print y-axis range usedprint(f"Global Y-axis range used for all panels: [{y_min:.2f}, {y_max:.2f}] pp")print("This allows direct visual comparison of training dynamics across all experiments.")

## 7. Summary Table

In [ ]:
# Create summary table
summary_data = []

for exp in experiments:
    exp_name = exp['name']
    if exp_name in final_performance:
        perf = final_performance[exp_name]
        summary_data.append({
            'Experiment': perf['label'],
            'Train MAE': f"{perf['train_mae']:.2f}",
            'Temporal MAE': f"{perf['temporal_mae']:.2f}",
            'Spatial MAE': f"{perf['spatial_mae']:.2f}",
            'Random MAE': f"{perf['random_mae']:.2f}"
        })

summary_df = pd.DataFrame(summary_data)

print("\n" + "="*100)
print(f"FINAL PERFORMANCE SUMMARY (After {EPOCHS} Epochs)")
print("="*100)
print(summary_df.to_string(index=False))
print("="*100)

# Save table
csv_path = os.path.join(OUTPUT_DIR, 'ablation_summary_table.csv')
summary_df.to_csv(csv_path, index=False)
print(f"\nSummary table saved to: {csv_path}")

### Cell 1: Extract Final Unique vs Multi-Species Performance

In [ ]:
import pandas as pdimport numpy as np# Extract final epoch metrics for unique and multi-species locationsunique_degen_results = {}for exp_name, exp_data in all_metrics.items():    df = exp_data['data']    final = df.iloc[-1]    unique_degen_results[exp_name] = {        'label': exp_data['label'],        # Training        'train_overall': final['train_mae'],        'train_unique': final['train_unique_mae'],        'train_degen': final['train_degen_mae'],        # Temporal        'temporal_overall': final['temporal_mae'],        'temporal_unique': final['temporal_unique_mae'],        'temporal_degen': final['temporal_degen_mae'],        # Spatial        'spatial_overall': final['spatial_mae'],        'spatial_unique': final['spatial_unique_mae'],        'spatial_degen': final['spatial_degen_mae'],        # Random        'random_overall': final['random_mae'],        'random_unique': final['random_unique_mae'],        'random_degen': final['random_degen_mae']    }print("Extracted unique vs multi-species metrics for all experiments")

### Cell 2: Create Comprehensive Comparison Table

In [ ]:
# Create detailed comparison tabletable_data = []for exp in experiments:    exp_name = exp['name']    if exp_name not in unique_degen_results:        continue    r = unique_degen_results[exp_name]    table_data.append({        'Experiment': r['label'],        # Training        'Train Overall': f"{r['train_overall']:.2f}",        'Train Unique': f"{r['train_unique']:.2f}",        'Train Multi': f"{r['train_degen']:.2f}",        'Train Δ': f"{r['train_degen'] - r['train_unique']:+.2f}",        # Temporal        'Temp Overall': f"{r['temporal_overall']:.2f}",        'Temp Unique': f"{r['temporal_unique']:.2f}",        'Temp Multi': f"{r['temporal_degen']:.2f}",        'Temp Δ': f"{r['temporal_degen'] - r['temporal_unique']:+.2f}",        # Spatial        'Spat Overall': f"{r['spatial_overall']:.2f}",        'Spat Unique': f"{r['spatial_unique']:.2f}",        'Spat Multi': f"{r['spatial_degen']:.2f}",        'Spat Δ': f"{r['spatial_degen'] - r['spatial_unique']:+.2f}",        # Random        'Rand Overall': f"{r['random_overall']:.2f}",        'Rand Unique': f"{r['random_unique']:.2f}",        'Rand Multi': f"{r['random_degen']:.2f}",        'Rand Δ': f"{r['random_degen'] - r['random_unique']:+.2f}"    })comparison_df = pd.DataFrame(table_data)print("\n" + "="*140)print("UNIQUE SPECIES LOCATIONS vs MULTI-SPECIES LOCATIONS - FINAL PERFORMANCE")print("="*140)print("MAE (percentage points)")print("Δ (Delta) = Multi-Species MAE - Unique MAE")print("  → Positive Δ: Multi-species locations are HARDER to predict")print("  → Negative Δ: Multi-species locations are EASIER to predict")print("="*140)print(comparison_df.to_string(index=False))print("="*140)# Save tablecsv_path = os.path.join(OUTPUT_DIR, 'unique_vs_multispecies_comparison.csv')comparison_df.to_csv(csv_path, index=False)print(f"\nTable saved to: {csv_path}")

### Cell 3: Figure - Unique vs Multi-Species Comparison (Bar Chart)

In [ ]:
import matplotlib.pyplot as pltimport numpy as npfig, axes = plt.subplots(2, 2, figsize=(20, 14))splits = [    ('train', 'Training Set', axes[0, 0]),    ('temporal', 'Temporal Test Set', axes[0, 1]),    ('spatial', 'Spatial Test Set', axes[1, 0]),    ('random', 'Random Test Set', axes[1, 1])]# Create abbreviated labels for experimentsdef create_short_label(label):    """Create short label with feature indicators"""    features = []    if 'Earth4D' in label:        features.append('E4D')    if 'AEF' in label:        features.append('AEF')    if 'Daymet' in label:        features.append('Day')    if 'Species' in label:        features.append('Sp')    if not features:        return 'None'    return '+'.join(features)exp_short_labels = [create_short_label(unique_degen_results[exp['name']]['label'])                    for exp in experiments if exp['name'] in unique_degen_results]for split_name, split_title, ax in splits:    # Extract data for this split    overall_vals = [unique_degen_results[exp['name']][f'{split_name}_overall']                    for exp in experiments if exp['name'] in unique_degen_results]    unique_vals = [unique_degen_results[exp['name']][f'{split_name}_unique']                   for exp in experiments if exp['name'] in unique_degen_results]    degen_vals = [unique_degen_results[exp['name']][f'{split_name}_degen']                  for exp in experiments if exp['name'] in unique_degen_results]    x = np.arange(len(exp_short_labels))    width = 0.25    # Plot bars    bars1 = ax.bar(x - width, overall_vals, width, label='Overall', color='#7F8C8D', alpha=0.8)    bars2 = ax.bar(x, unique_vals, width, label='Unique Species Locations', color='#27AE60', alpha=0.8)    bars3 = ax.bar(x + width, degen_vals, width, label='Multi-Species Locations', color='#E74C3C', alpha=0.8)    # Formatting    ax.set_xlabel('Experiment', fontsize=12, fontweight='bold')    ax.set_ylabel('MAE (percentage points)', fontsize=12, fontweight='bold')    ax.set_title(split_title, fontsize=14, fontweight='bold', pad=15)    ax.set_xticks(x)    ax.set_xticklabels(exp_short_labels, fontsize=10, rotation=45, ha='right')    ax.legend(loc='upper right', fontsize=10, framealpha=0.95)    ax.grid(True, alpha=0.3, axis='y', linestyle='--')    # Add value labels on bars for unique and multi-species    for i, (u, d) in enumerate(zip(unique_vals, degen_vals)):        # Only label if there's a meaningful difference        diff = d - u        if abs(diff) > 0.5:            ax.text(i, max(u, d) + 0.5, f'Δ{diff:+.1f}', ha='center', va='bottom',                   fontsize=8, color='red' if diff > 0 else 'green', fontweight='bold')fig.suptitle('Unique Species Locations vs Multi-Species Locations - All Experiments\n(Δ = Multi-Species MAE - Unique MAE)',             fontsize=16, fontweight='bold', y=0.995)plt.tight_layout()# Savepng_path = os.path.join(OUTPUT_DIR, 'fig_unique_vs_multispecies_bars.png')svg_path = os.path.join(OUTPUT_DIR, 'fig_unique_vs_multispecies_bars.svg')plt.savefig(png_path, dpi=300, bbox_inches='tight')plt.savefig(svg_path, dpi=300, bbox_inches='tight')print(f"Saved: {png_path}")print(f"Saved: {svg_path}")plt.show()

### Cell 4: Figure - Learning Curves (Unique vs Multi-Species)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22, 10))axes = axes.flatten()for idx, exp in enumerate(experiments):    exp_name = exp['name']    if exp_name not in all_metrics:        continue    ax = axes[idx]    df = all_metrics[exp_name]['data']    # Plot temporal test learning curves (unique vs multi-species)    ax.plot(df['epoch'], df['temporal_unique_mae'],            label='Unique Locations', color='#27AE60', linewidth=2, alpha=0.9)    ax.plot(df['epoch'], df['temporal_degen_mae'],            label='Multi-Species Locations', color='#E74C3C', linewidth=2, alpha=0.9)    # Add overall for reference    ax.plot(df['epoch'], df['temporal_mae'],            label='Overall', color='#7F8C8D', linewidth=1.5, linestyle='--', alpha=0.6)    # Create short label    short_label = create_short_label(unique_degen_results[exp_name]['label'])    # Formatting    ax.set_xlabel('Epoch', fontsize=10, fontweight='bold')    ax.set_ylabel('Temporal Test MAE (pp)', fontsize=10, fontweight='bold')    ax.set_title(f"{idx+1}. {short_label}",                fontsize=12, fontweight='bold')    ax.legend(loc='upper right', fontsize=8)    ax.grid(True, alpha=0.3, linestyle='--')    ax.set_xlim(0, EPOCHS)    # Add final values as text    final_unique = df['temporal_unique_mae'].iloc[-1]    final_degen = df['temporal_degen_mae'].iloc[-1]    diff = final_degen - final_unique    ax.text(0.98, 0.02, f'Δ = {diff:+.1f}pp',           transform=ax.transAxes, ha='right', va='bottom',           fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),           fontweight='bold', color='red' if diff > 0 else 'green')fig.suptitle('Temporal Test Learning Curves: Unique vs Multi-Species Locations\n(Δ = Multi-Species MAE - Unique MAE)',             fontsize=16, fontweight='bold', y=0.995)plt.tight_layout()# Savepng_path = os.path.join(OUTPUT_DIR, 'fig_unique_vs_multispecies_curves.png')svg_path = os.path.join(OUTPUT_DIR, 'fig_unique_vs_multispecies_curves.svg')plt.savefig(png_path, dpi=300, bbox_inches='tight')plt.savefig(svg_path, dpi=300, bbox_inches='tight')print(f"Saved: {png_path}")print(f"Saved: {svg_path}")plt.show()

### Cell 5: Summary Analysis - Key Insights

In [ ]:
print("\n" + "="*100)print("KEY INSIGHTS: UNIQUE vs MULTI-SPECIES LOCATIONS")print("="*100)# Calculate average difference across experimentsavg_diffs = {    'train': [],    'temporal': [],    'spatial': [],    'random': []}for exp_name in unique_degen_results:    r = unique_degen_results[exp_name]    avg_diffs['train'].append(r['train_degen'] - r['train_unique'])    avg_diffs['temporal'].append(r['temporal_degen'] - r['temporal_unique'])    avg_diffs['spatial'].append(r['spatial_degen'] - r['spatial_unique'])    avg_diffs['random'].append(r['random_degen'] - r['random_unique'])print("\n1. AVERAGE DIFFICULTY INCREASE (Multi-Species vs Unique):")print(f"   Training:  {np.mean(avg_diffs['train']):+.2f} pp (±{np.std(avg_diffs['train']):.2f})")print(f"   Temporal:  {np.mean(avg_diffs['temporal']):+.2f} pp (±{np.std(avg_diffs['temporal']):.2f})")print(f"   Spatial:   {np.mean(avg_diffs['spatial']):+.2f} pp (±{np.std(avg_diffs['spatial']):.2f})")print(f"   Random:    {np.mean(avg_diffs['random']):+.2f} pp (±{np.std(avg_diffs['random']):.2f})")# Find experiments where multi-species performs BETTER (negative diff)print("\n2. EXPERIMENTS WHERE MULTI-SPECIES PERFORMS BETTER THAN UNIQUE:")found_any = Falsefor exp_name, r in unique_degen_results.items():    improvements = []    if r['temporal_degen'] < r['temporal_unique']:        improvements.append(f"Temporal: {r['temporal_degen'] - r['temporal_unique']:.2f}pp")    if r['spatial_degen'] < r['spatial_unique']:        improvements.append(f"Spatial: {r['spatial_degen'] - r['spatial_unique']:.2f}pp")    if r['random_degen'] < r['random_unique']:        improvements.append(f"Random: {r['random_degen'] - r['random_unique']:.2f}pp")    if improvements:        found_any = True        print(f"   {r['label'][:50]}")        for imp in improvements:            print(f"     - {imp}")if not found_any:    print("   None - Multi-species locations are consistently harder across all experiments")# Best models for each location typeprint("\n3. BEST MODELS BY LOCATION TYPE (Temporal Test):")# Unique locationsbest_unique = min(unique_degen_results.items(),                 key=lambda x: x[1]['temporal_unique'])print(f"   Unique Locations:        {best_unique[1]['label'][:50]}")print(f"                            MAE = {best_unique[1]['temporal_unique']:.2f}pp")# Multi-species locationsbest_degen = min(unique_degen_results.items(),                key=lambda x: x[1]['temporal_degen'])print(f"   Multi-Species Locations: {best_degen[1]['label'][:50]}")print(f"                            MAE = {best_degen[1]['temporal_degen']:.2f}pp")# Overallbest_overall = min(unique_degen_results.items(),                  key=lambda x: x[1]['temporal_overall'])print(f"   Overall:                 {best_overall[1]['label'][:50]}")print(f"                            MAE = {best_overall[1]['temporal_overall']:.2f}pp")print("\n4. SPECIES EMBEDDING IMPACT ON DEGENERACY:")# Compare Earth4D+AEF with and without species for degenerate samplese3 = unique_degen_results['exp3_earth4d_aef']e7 = unique_degen_results['exp7_earth4d_aef_species']print(f"   Earth4D + AEF (no species):")print(f"     Temporal Multi-Species MAE: {e3['temporal_degen']:.2f}pp")print(f"   Earth4D + AEF + Species:")print(f"     Temporal Multi-Species MAE: {e7['temporal_degen']:.2f}pp")print(f"   → Improvement: {e3['temporal_degen'] - e7['temporal_degen']:.2f}pp ({100*(e3['temporal_degen'] - e7['temporal_degen'])/e3['temporal_degen']:.1f}%)")# ==============================================================================# NEW INSIGHT 5: EARTH4D WITH AND WITHOUT AEF - DETAILED COMPARISON# ==============================================================================print("\n5. EARTH4D WITH AND WITHOUT AEF - DETAILED COMPARISON:")print("   " + "="*90)# Get experimentse1 = unique_degen_results['exp1_earth4d_only']          # Earth4D Onlye3 = unique_degen_results['exp3_earth4d_aef']           # Earth4D + AEFe5 = unique_degen_results['exp5_earth4d_species']       # Earth4D + Speciese7 = unique_degen_results['exp7_earth4d_aef_species']   # Earth4D + AEF + Speciesprint("\n   A) WITHOUT SPECIES EMBEDDINGS:")print("      Baseline: Earth4D Only")print("      Comparison: Earth4D + AEF")print("      " + "-"*86)# Training Setprint(f"\n      TRAINING SET:")print(f"        Earth4D Only:")print(f"          Overall:      {e1['train_overall']:6.2f} pp  |  Unique: {e1['train_unique']:6.2f} pp  |  Multi: {e1['train_degen']:6.2f} pp")print(f"        Earth4D + AEF:")print(f"          Overall:      {e3['train_overall']:6.2f} pp  |  Unique: {e3['train_unique']:6.2f} pp  |  Multi: {e3['train_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF - E4D):")print(f"          Overall:      {e3['train_overall'] - e1['train_overall']:+6.2f} pp  |  Unique: {e3['train_unique'] - e1['train_unique']:+6.2f} pp  |  Multi: {e3['train_degen'] - e1['train_degen']:+6.2f} pp")# Temporal Testprint(f"\n      TEMPORAL TEST:")print(f"        Earth4D Only:")print(f"          Overall:      {e1['temporal_overall']:6.2f} pp  |  Unique: {e1['temporal_unique']:6.2f} pp  |  Multi: {e1['temporal_degen']:6.2f} pp")print(f"        Earth4D + AEF:")print(f"          Overall:      {e3['temporal_overall']:6.2f} pp  |  Unique: {e3['temporal_unique']:6.2f} pp  |  Multi: {e3['temporal_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF - E4D):")print(f"          Overall:      {e3['temporal_overall'] - e1['temporal_overall']:+6.2f} pp  |  Unique: {e3['temporal_unique'] - e1['temporal_unique']:+6.2f} pp  |  Multi: {e3['temporal_degen'] - e1['temporal_degen']:+6.2f} pp")# Spatial Testprint(f"\n      SPATIAL TEST:")print(f"        Earth4D Only:")print(f"          Overall:      {e1['spatial_overall']:6.2f} pp  |  Unique: {e1['spatial_unique']:6.2f} pp  |  Multi: {e1['spatial_degen']:6.2f} pp")print(f"        Earth4D + AEF:")print(f"          Overall:      {e3['spatial_overall']:6.2f} pp  |  Unique: {e3['spatial_unique']:6.2f} pp  |  Multi: {e3['spatial_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF - E4D):")print(f"          Overall:      {e3['spatial_overall'] - e1['spatial_overall']:+6.2f} pp  |  Unique: {e3['spatial_unique'] - e1['spatial_unique']:+6.2f} pp  |  Multi: {e3['spatial_degen'] - e1['spatial_degen']:+6.2f} pp")# Random Testprint(f"\n      RANDOM TEST:")print(f"        Earth4D Only:")print(f"          Overall:      {e1['random_overall']:6.2f} pp  |  Unique: {e1['random_unique']:6.2f} pp  |  Multi: {e1['random_degen']:6.2f} pp")print(f"        Earth4D + AEF:")print(f"          Overall:      {e3['random_overall']:6.2f} pp  |  Unique: {e3['random_unique']:6.2f} pp  |  Multi: {e3['random_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF - E4D):")print(f"          Overall:      {e3['random_overall'] - e1['random_overall']:+6.2f} pp  |  Unique: {e3['random_unique'] - e1['random_unique']:+6.2f} pp  |  Multi: {e3['random_degen'] - e1['random_degen']:+6.2f} pp")print("\n   " + "-"*90)print("\n   B) WITH SPECIES EMBEDDINGS:")print("      Baseline: Earth4D + Species")print("      Comparison: Earth4D + AEF + Species")print("      " + "-"*86)# Training Setprint(f"\n      TRAINING SET:")print(f"        Earth4D + Species:")print(f"          Overall:      {e5['train_overall']:6.2f} pp  |  Unique: {e5['train_unique']:6.2f} pp  |  Multi: {e5['train_degen']:6.2f} pp")print(f"        Earth4D + AEF + Species:")print(f"          Overall:      {e7['train_overall']:6.2f} pp  |  Unique: {e7['train_unique']:6.2f} pp  |  Multi: {e7['train_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF+Sp - E4D+Sp):")print(f"          Overall:      {e7['train_overall'] - e5['train_overall']:+6.2f} pp  |  Unique: {e7['train_unique'] - e5['train_unique']:+6.2f} pp  |  Multi: {e7['train_degen'] - e5['train_degen']:+6.2f} pp")# Temporal Testprint(f"\n      TEMPORAL TEST:")print(f"        Earth4D + Species:")print(f"          Overall:      {e5['temporal_overall']:6.2f} pp  |  Unique: {e5['temporal_unique']:6.2f} pp  |  Multi: {e5['temporal_degen']:6.2f} pp")print(f"        Earth4D + AEF + Species:")print(f"          Overall:      {e7['temporal_overall']:6.2f} pp  |  Unique: {e7['temporal_unique']:6.2f} pp  |  Multi: {e7['temporal_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF+Sp - E4D+Sp):")print(f"          Overall:      {e7['temporal_overall'] - e5['temporal_overall']:+6.2f} pp  |  Unique: {e7['temporal_unique'] - e5['temporal_unique']:+6.2f} pp  |  Multi: {e7['temporal_degen'] - e5['temporal_degen']:+6.2f} pp")# Spatial Testprint(f"\n      SPATIAL TEST:")print(f"        Earth4D + Species:")print(f"          Overall:      {e5['spatial_overall']:6.2f} pp  |  Unique: {e5['spatial_unique']:6.2f} pp  |  Multi: {e5['spatial_degen']:6.2f} pp")print(f"        Earth4D + AEF + Species:")print(f"          Overall:      {e7['spatial_overall']:6.2f} pp  |  Unique: {e7['spatial_unique']:6.2f} pp  |  Multi: {e7['spatial_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF+Sp - E4D+Sp):")print(f"          Overall:      {e7['spatial_overall'] - e5['spatial_overall']:+6.2f} pp  |  Unique: {e7['spatial_unique'] - e5['spatial_unique']:+6.2f} pp  |  Multi: {e7['spatial_degen'] - e5['spatial_degen']:+6.2f} pp")# Random Testprint(f"\n      RANDOM TEST:")print(f"        Earth4D + Species:")print(f"          Overall:      {e5['random_overall']:6.2f} pp  |  Unique: {e5['random_unique']:6.2f} pp  |  Multi: {e5['random_degen']:6.2f} pp")print(f"        Earth4D + AEF + Species:")print(f"          Overall:      {e7['random_overall']:6.2f} pp  |  Unique: {e7['random_unique']:6.2f} pp  |  Multi: {e7['random_degen']:6.2f} pp")print(f"        → Improvement (E4D+AEF+Sp - E4D+Sp):")print(f"          Overall:      {e7['random_overall'] - e5['random_overall']:+6.2f} pp  |  Unique: {e7['random_unique'] - e5['random_unique']:+6.2f} pp  |  Multi: {e7['random_degen'] - e5['random_degen']:+6.2f} pp")# Summaryprint("\n   " + "-"*90)print("\n   C) KEY FINDINGS:")# Calculate average improvementsno_species_improvements = {    'train': e3['train_overall'] - e1['train_overall'],    'temporal': e3['temporal_overall'] - e1['temporal_overall'],    'spatial': e3['spatial_overall'] - e1['spatial_overall'],    'random': e3['random_overall'] - e1['random_overall']}with_species_improvements = {    'train': e7['train_overall'] - e5['train_overall'],    'temporal': e7['temporal_overall'] - e5['temporal_overall'],    'spatial': e7['spatial_overall'] - e5['spatial_overall'],    'random': e7['random_overall'] - e5['random_overall']}print(f"\n      • WITHOUT Species Embeddings:")print(f"        AEF provides LARGE improvement on spatial test: {no_species_improvements['spatial']:+.2f} pp")print(f"        Modest impact on temporal: {no_species_improvements['temporal']:+.2f} pp")print(f"        Average improvement across test sets: {np.mean([no_species_improvements['temporal'], no_species_improvements['spatial'], no_species_improvements['random']]):+.2f} pp")print(f"\n      • WITH Species Embeddings:")print(f"        AEF provides consistent improvement on temporal: {with_species_improvements['temporal']:+.2f} pp")print(f"        Slight increase on spatial: {with_species_improvements['spatial']:+.2f} pp")print(f"        Average improvement across test sets: {np.mean([with_species_improvements['temporal'], with_species_improvements['spatial'], with_species_improvements['random']]):+.2f} pp")print(f"\n      • AEF Impact on Multi-Species Locations:")print(f"        Without species: {e3['temporal_degen'] - e1['temporal_degen']:+.2f} pp (temporal)")print(f"        With species: {e7['temporal_degen'] - e5['temporal_degen']:+.2f} pp (temporal)")print(f"\n      • CONCLUSION:")if abs(no_species_improvements['spatial']) > abs(with_species_improvements['spatial']):    print(f"        AEF provides GREATER benefit when species information is NOT available")    print(f"        Spatial improvement: {no_species_improvements['spatial']:+.2f} pp without species vs {with_species_improvements['spatial']:+.2f} pp with species")else:    print(f"        AEF provides consistent benefit regardless of species information")    print(f"        Best performance: Earth4D + AEF + Species (13.50 pp temporal MAE)")print("\n" + "="*100)print("Analysis complete! Check the output directory for detailed figures.")print("="*100)

### Cell 6: Training vs Test Loss Over Time (Learning Curves)

In [ ]:
import matplotlib.pyplot as pltimport numpy as npfig, axes = plt.subplots(2, 4, figsize=(24, 12))axes = axes.flatten()for idx, exp in enumerate(experiments):    exp_name = exp['name']    if exp_name not in all_metrics:        continue    ax = axes[idx]    df = all_metrics[exp_name]['data']    # Create short label    short_label = create_short_label(all_metrics[exp_name]['label'])    # Plot training loss    ax.plot(df['epoch'], df['train_mae'],            label='Training', color='#2C3E50', linewidth=2.5, alpha=0.9)    # Plot test losses    ax.plot(df['epoch'], df['temporal_mae'],            label='Temporal Test', color='#E74C3C', linewidth=2, alpha=0.9)    ax.plot(df['epoch'], df['spatial_mae'],            label='Spatial Test', color='#3498DB', linewidth=2, alpha=0.9)    ax.plot(df['epoch'], df['random_mae'],            label='Random Test', color='#2ECC71', linewidth=2, alpha=0.9)    # Formatting    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')    ax.set_ylabel('MAE (percentage points)', fontsize=11, fontweight='bold')    ax.set_title(f"{idx+1}. {short_label}",                fontsize=13, fontweight='bold', pad=10)    ax.legend(loc='upper right', fontsize=9, framealpha=0.95)    ax.grid(True, alpha=0.3, linestyle='--')    ax.set_xlim(0, EPOCHS)    # Add final values as text    final_train = df['train_mae'].iloc[-1]    final_temporal = df['temporal_mae'].iloc[-1]    final_spatial = df['spatial_mae'].iloc[-1]    final_random = df['random_mae'].iloc[-1]    # Calculate generalization gap (temporal test - training)    gen_gap = final_temporal - final_train    # Add text box with final values    textstr = f'Final Epoch {EPOCHS}:\n'    textstr += f'Train: {final_train:.2f} pp\n'    textstr += f'Temporal: {final_temporal:.2f} pp\n'    textstr += f'Spatial: {final_spatial:.2f} pp\n'    textstr += f'Random: {final_random:.2f} pp\n'    textstr += f'Gen Gap: {gen_gap:+.2f} pp'    props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)    ax.text(0.02, 0.97, textstr,           transform=ax.transAxes, fontsize=8,           verticalalignment='top', horizontalalignment='left',           bbox=props, family='monospace')fig.suptitle('Training vs Test Loss Over Time - All Experiments\n(Gen Gap = Temporal Test MAE - Training MAE)',             fontsize=17, fontweight='bold', y=0.995)plt.tight_layout()# Savepng_path = os.path.join(OUTPUT_DIR, 'fig_training_vs_test_loss.png')svg_path = os.path.join(OUTPUT_DIR, 'fig_training_vs_test_loss.svg')plt.savefig(png_path, dpi=300, bbox_inches='tight')plt.savefig(svg_path, dpi=300, bbox_inches='tight')print(f"\nSaved: {png_path}")print(f"Saved: {svg_path}")plt.show()# Print generalization gap analysisprint("\n" + "="*100)print("GENERALIZATION GAP ANALYSIS (Temporal Test MAE - Training MAE)")print("="*100)print(f"{'Experiment':<50} {'Train MAE':<12} {'Temporal MAE':<15} {'Gen Gap':<12}")print("-"*100)gen_gaps = []for exp in experiments:    exp_name = exp['name']    if exp_name not in all_metrics:        continue    df = all_metrics[exp_name]['data']    final = df.iloc[-1]    train_mae = final['train_mae']    temporal_mae = final['temporal_mae']    gap = temporal_mae - train_mae    gen_gaps.append((all_metrics[exp_name]['label'], gap))    short_label = create_short_label(all_metrics[exp_name]['label'])    print(f"{short_label:<50} {train_mae:>10.2f} pp  {temporal_mae:>13.2f} pp  {gap:>10.2f} pp")print("-"*100)avg_gap = np.mean([g[1] for g in gen_gaps])print(f"{'Average Generalization Gap:':<50} {avg_gap:>39.2f} pp")print("="*100)# Find best and worst generalizationbest_gen = min(gen_gaps, key=lambda x: x[1])worst_gen = max(gen_gaps, key=lambda x: x[1])print(f"\nBest Generalization (smallest gap):  {create_short_label(best_gen[0])} ({best_gen[1]:.2f} pp)")print(f"Worst Generalization (largest gap):  {create_short_label(worst_gen[0])} ({worst_gen[1]:.2f} pp)")print("="*100)

## 8. Package and Download Results

In [ ]:
import zipfile
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_name = f"ablation_study_results_{timestamp}.zip"
zip_path = os.path.join('/content', zip_name)

print(f"Creating results package: {zip_name}")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    # Add all plots (PNG and SVG)
    for file in os.listdir(OUTPUT_DIR):
        if file.endswith(('.png', '.svg', '.csv')):
            file_path = os.path.join(OUTPUT_DIR, file)
            zipf.write(file_path, file)
            print(f"  Added: {file}")
    
    # Add metrics from all experiments
    for exp in experiments:
        exp_dir = os.path.join(OUTPUT_DIR, exp['name'])
        if os.path.exists(exp_dir):
            for file in os.listdir(exp_dir):
                if file.endswith('.csv'):
                    file_path = os.path.join(exp_dir, file)
                    arc_name = os.path.join(exp['name'], file)
                    zipf.write(file_path, arc_name)
                    print(f"  Added: {arc_name}")

zip_size = os.path.getsize(zip_path) / (1024*1024)
print(f"\nResults package created: {zip_name} ({zip_size:.1f} MB)")

# Download
from google.colab import files
files.download(zip_path)
print("Download started!")

## 9. Final Summary

In [ ]:
print("\n" + "="*100)
print("ABLATION STUDY COMPLETE")
print("="*100)
print(f"\nConfiguration:")
print(f"  Epochs per experiment: {EPOCHS}")
print(f"  Species embedding dimension: {SPECIES_DIM}D")
print(f"  Total experiments: 8")
print(f"\nExperiments completed:")
for i, exp in enumerate(experiments, 1):
    print(f"  {i}. {exp['label']}")
print(f"\nFigures generated:")
print(f"  â€¢ Figure 1: Training loss curves (all experiments)")
print(f"  â€¢ Figure 2: Test performance (temporal, spatial, random)")
print(f"  â€¢ Figure 3: Final performance comparison (bar chart)")
print(f"  â€¢ Figure 4: Species impact comparison")
print(f"\nAll figures saved in:")
print(f"  â€¢ PNG format (DPI 300)")
print(f"  â€¢ SVG format (DPI 300, vector graphics)")
print(f"\nResults location: {OUTPUT_DIR}")
print(f"Download package: {zip_name}")
print("="*100)